In [1]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

/home/aifather/venv-qwen3tts/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.is_available()

True

In [3]:
model_id1= "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
model_id2= "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model_id3= "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
model_id4=  "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"

In [4]:
AudioPATH= "./audio"
RefVoicePATH1 = "./audio/ref_voice/warm_reporter_well_paced_steady_friendly.mp3"
RefVoicePATHWav1 = "./audio/ref_voice/warm_reporter_well_paced_steady_friendly.wav"
RefVoicePATHWav2 =f"{AudioPATH}/ref_voice/alienkevin.wav"
referTxt2= "昨晚我嘗試咗去做一個vegetarian pizza，雖然冇咗肉，但係蕃茄醬好rich，起司又melt得剛剛好！"

In [5]:
!huggingface-cli download Qwen/Qwen3-TTS-12Hz-0.6B-Base --local-dir ./Qwen3-TTS-12Hz-0.6B-Base
!huggingface-cli download Qwen/Qwen3-TTS-12Hz-1.7B-Base --local-dir ./Qwen3-TTS-12Hz-1.7B-Base
!huggingface-cli download Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign --local-dir ./Qwen3-TTS-12Hz-1.7B-VoiceDesign
!huggingface-cli download Qwen/Qwen3-ASR-0.6B --local-dir ./Qwen3-ASR-0.6B

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 1581.47it/s]
/home/aifather/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training/Qwen3-TTS-12Hz-0.6B-Base
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 1311.07it/s]
/home/aifather/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training/Qwen3-TTS-12Hz-1.7B-Base
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 1353.07it/s]
/home/aifather/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training/Qwen3-TTS-12Hz-1.7B-VoiceDesign
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files: 100%|██████████████████████| 10/10 [00:00<00:00, 1565.74it/s]
/home/aifather/qwen3-tts-hk-cantonese-finetu

In [61]:
RefVoicePATHWav2RefVoicePATHWav2

NameError: name 'RefVoicePATHWav2RefVoicePATHWav2' is not defined

In [6]:

model = Qwen3TTSModel.from_pretrained(
    model_id2,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2", # for nvidia ampere or above archiect
    attn_implementation= "sdpa",  # no flash attention 2 for v100
)


Fetching 4 files: 100%|████████████████████████| 4/4 [00:00<00:00, 18872.01it/s]


In [7]:
model

In [55]:

# Custom Voice Generate Pre-set 9 personal voice
# %%time
# # only for  CustomVoice model used
# wavs, sr = model.generate_custom_voice(
#     text="其实我真的有发现，我是一个特别善于观察别人情绪的人。",
#     language="Chinese", # Pass `Auto` (or omit) for auto language adaptive; if the target language is known, set it explicitly.
#     speaker="Vivian",
#     # instruct="用特别愤怒的语气说", # Omit if not needed.
#     instruct="用自然的香港粤语语气说",
# )
# sf.write("output_custom_voice.wav", wavs[0], sr)

## Voice Clone

In [8]:
%%time
# for ba
wavs, sr = model.generate_voice_clone(
    text="飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。",
    language="chinese",
    ref_audio=RefVoicePATHWav2,
    ref_text=referTxt2,
    instruct="用特别愤怒的语气说",
)
sf.write("output_voice_clone.wav", wavs[0], sr)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[WARNING] Min value of input waveform signal is -1.0005865097045898
[WARNING] Max value of input waveform signal is 1.0000923871994019
CPU times: user 16.3 s, sys: 309 ms, total: 16.6 s
Wall time: 16.6 s


In [42]:
%%time
!python qwen3_voice_clone_cli.py \
  --reference_audio $"{AudioPATH}/ref_voice/alienkevin.wav" \
  --generate_text "今天天气很好！但天氣好熱,感覺似要落大雨" \
  --qwen3_asr_model ./Qwen3-ASR-0.6B \
  --qwen3_tts_model ./Qwen3-TTS-12Hz-1.7B-Base \
  --no-flash-attn \
  --language Chinese \
  --temperature 0.8 \
  --top_p 0.95 \
  --output_path chinese_cloned.wav
 

Loading Qwen3-ASR model: ./Qwen3-ASR-0.6B on cuda...
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading Qwen3-TTS model: ./Qwen3-TTS-12Hz-1.7B-Base on cuda...
   → Flash Attention 2: DISABLED (older GPU or --no-flash-attn)
talker_config is None. Initializing talker model with default values
speaker_encoder_config is None. Initializing talker model with default values
code_predictor_config is None. Initializing code_predictor model with default values
code_predictor_config is None. Initializing code_predictor model with default values
encoder_config is None. Initializing encoder with default values
decoder_config is None. Initializing decoder with default values
🔊 Transcribing reference audio with ./Qwen3-ASR-0.6B...
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
✅ Auto-generated ref_text: 昨晚我嘗試咗去做一個 vegetarian pizza，雖然冇咗肉，但係番茄醬好rich，皮絲有melts得剛剛好。
🎤 Language: Chinese
🎙️ C

In [32]:
RefVoicePATHWav2

'./audio/ref_voice/alienkevin.wav'

In [ ]:
!python voice_clone_server.py

🌐 Starting Qwen3 Voice Clone Web Server...
* Running on local URL:  http://0.0.0.0:7860
HTTP Request: GET http://localhost:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
HTTP Request: HEAD http://localhost:7860/ "HTTP/1.1 200 OK"
* To create a public link, set `share=True` in `launch()`.
HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
🚀 Loading Qwen3-ASR: ./Qwen3-ASR-0.6B on cuda...
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
🚀 Loading Qwen3-TTS: Qwen/Qwen3-TTS-12Hz-1.7B-Base on cuda → Flash Attn: OFF
talker_config is None. Initializing talker model with default values
speaker_encoder_config is None. Initializing talker model with default values
code_predictor_config is None. Initializing code_predictor model with default values
code_predictor_config is None. Initializing code_predictor model with default values
Fetching 4 files: 100%|████████████████████████| 4/4 [00:00<0